In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('house_price_prediction_dataset.csv')

In [4]:
df.shape

(5000, 14)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_type               5000 non-null   str    
 1   neighborhood                5000 non-null   str    
 2   square_feet                 4879 non-null   float64
 3   lot_square_feet             4588 non-null   float64
 4   bedrooms                    4822 non-null   float64
 5   bathrooms                   4741 non-null   float64
 6   year_built                  4727 non-null   float64
 7   condition                   4649 non-null   str    
 8   heating_type                4253 non-null   str    
 9   parking_spaces              4598 non-null   float64
 10  has_garden                  4767 non-null   str    
 11  renovated                   4699 non-null   str    
 12  distance_to_city_center_km  4789 non-null   float64
 13  sale_price_usd              5000 non-null   

In [6]:
df.describe()

,square_feet,lot_square_feet,bedrooms,bathrooms,year_built,parking_spaces,distance_to_city_center_km,sale_price_usd
count,4879.000000,4588.000000,4822.000000,4741.000000,4727.000000,4598.000000,4789.000000,5000.000000
mean,1876.759787,5608.369878,3.436748,2.376819,1981.957267,1.559809,6.971622,452508.586800
std,639.047207,3098.176587,1.401262,0.914840,24.848297,0.825293,4.643813,139966.649805
min,450.000000,800.000000,1.000000,1.000000,1940.000000,0.000000,0.200000,45000.000000
25%,1438.000000,3270.000000,2.000000,1.500000,1961.000000,1.000000,3.500000,355822.500000
50%,1867.000000,5195.000000,3.000000,2.500000,1981.000000,2.000000,6.000000,451563.000000
75%,2303.500000,7570.750000,4.000000,3.000000,2004.000000,2.000000,9.400000,547531.000000
max,4162.000000,18604.000000,7.000000,5.000000,2025.000000,3.000000,35.000000,977217.000000


In [7]:
df.isnull().sum()

property_type                   0
neighborhood                    0
square_feet                   121
lot_square_feet               412
bedrooms                      178
bathrooms                     259
year_built                    273
condition                     351
heating_type                  747
parking_spaces                402
has_garden                    233
renovated                     301
distance_to_city_center_km    211
sale_price_usd                  0
dtype: int64

In [8]:
# 2. How are prices distributed?
print(df["sale_price_usd"].describe())

count      5000.000000
mean     452508.586800
std      139966.649805
min       45000.000000
25%      355822.500000
50%      451563.000000
75%      547531.000000
max      977217.000000
Name: sale_price_usd, dtype: float64


In [9]:
# 3. Does size drive price? (the single strongest signal usually)
print(df[["square_feet", "sale_price_usd"]].corr())

                square_feet  sale_price_usd
square_feet         1.00000         0.89985
sale_price_usd      0.89985         1.00000


In [12]:
# Make sure 'None' in heating_type is treated as a real word, not a blank
df['heating_type'] = df['heating_type'].astype(str).replace('nan', np.nan)

In [16]:
# 2. Simple Feature Engineering
df["house_age"] = 2026 - df["year_built"]  # 2026 is the current year in your data
df = df.drop("year_built", axis=1)


KeyError: 'year_built'

In [17]:
print(df["condition"].dtype)
print(df["condition"].unique())

float64
[ 3.  2.  4.  5. nan  1.]


In [15]:
# Map 'condition' to numbers (1 to 5)

#  ---- method 1: using a dictionary and map
# condition_map = {"Poor": 1, "Fair": 2, "Good": 3, "Very Good": 4, "Excellent": 5}
# df["condition"] = df["condition"].map(condition_map)
# df["condition"] = df["condition"].replace(condition_map)

# --- method 2: using categorical type and codes

# levels = ["Poor", "Fair", "Good", "Very Good", "Excellent"]
# condition_type = pd.CategoricalDtype(categories=levels, ordered=True)

# df["condition"] = df["condition"].astype(condition_type).cat.codes + 1


#  --- method 3: using sklearn's OrdinalEncoder

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(
    categories=[["Poor", "Fair", "Good", "Very Good", "Excellent"]]
)

df[["condition"]] = encoder.fit_transform(df[["condition"]]) + 1

ValueError: could not convert string to float: 'Poor'

In [18]:

# Map Yes/No to 1/0

#  ---- method 1: using a dictionary and map
# df['has_garden'] = df['has_garden'].map({'Yes': 1, 'No': 0})
# df['renovated'] = df['renovated'].map({'Yes': 1, 'No': 0})

#  --- method 1.2: using replace

yes_no_map = {"Yes": 1, "No": 0}
df["has_garden"] = df["has_garden"].replace(yes_no_map)
df["renovated"] = df["renovated"].replace(yes_no_map)


#  ---- method 2: using boolean comparison and astype
# df["has_garden"] = df["has_garden"].eq("Yes").astype("int8")
# df["renovated"] = df["renovated"].eq("Yes").astype("int8")

In [ ]:
x = df.drop("sale_price_usd", axis=1)
y = df["sale_price_usd"]

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
# 5. Group the columns by type
numeric_features = ['square_feet', 'lot_square_feet', 'bedrooms', 'bathrooms', 
                    'parking_spaces', 'distance_to_city_center_km', 'house_age', 'condition']
categorical_features = ['property_type', 'neighborhood', 'heating_type']